In [ ]:
# ============================================================
# INFERENCE — ECG Sleep Apnea Classification
# Input  : raw ECG segment (60s, 200Hz)
# Output : prediction (Normal/AH) + probability
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter
import neurokit2 as nk
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS         = 200
INTERP_FS  = 4
SEG_LEN    = 240
THRESHOLD  = 0.4472
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = r'E:\1D CNN\1d_cnn_fe\models\model_6feat_fold2.pth'

# Auto-create folders
os.makedirs(r'E:\1D CNN\1d_cnn_fe\models', exist_ok=True)

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel//reduction, channel, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        ch = out_ch // 4
        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))
    def forward(self, x):
        identity = self.shortcut(x)
        out = torch.cat([c(x) for c in self.conv1_list], dim=1)
        out = self.relu(self.bn1(out))
        out = torch.cat([c(out) for c in self.conv2_list], dim=1)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        return self.relu(out)

class MultiScaleSEResNet(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, 1)
        self.drop = nn.Dropout(0.5)
    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.gap(x).squeeze(-1)
        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# PREPROCESSING
# ============================================================
def bandpass_filter(ecg, fs=FS):
    """Bandpass filter 8-50Hz — same as training pipeline"""
    return nk.signal_filter(
        ecg,
        sampling_rate=fs,
        lowcut=8.0,
        highcut=50.0,
        method='butterworth'
    )

def zscore_normalize(ecg):
    """Z-score per segment — same as training pipeline"""
    mean = np.mean(ecg)
    std  = np.std(ecg)
    return (ecg - mean) / (std + 1e-8)

# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_6_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4:
            return None

        rr_ms    = np.diff(r_peaks) / FS * 1000.0
        rp_amp   = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS

        qrs_amps = []
        for i in range(len(r_peaks)):
            start = max(0, r_peaks[i] - int(0.08 * FS))
            end   = min(len(ecg), r_peaks[i] + int(0.08 * FS))
            qrs_amps.append(np.max(ecg[start:end]) - np.min(ecg[start:end]))
        qrs_amp_std = np.std(qrs_amps)

        mean_rr   = np.mean(rr_ms)
        mean_hr   = 60000 / mean_rr if mean_rr > 0 else 0
        hist, _   = np.histogram(rr_ms, bins=50)
        tri_index = len(rr_ms) / np.max(hist) if np.max(hist) > 0 else 0
        nn50      = np.sum(np.abs(np.diff(rr_ms)) > 50)
        pnn50     = (nn50 / len(rr_ms)) * 100 if len(rr_ms) > 0 else 0

        t_uni   = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        rr_uni  = interp1d(rr_times, rr_ms,  kind='cubic', fill_value='extrapolate')(t_uni)
        rp_uni  = interp1d(rr_times, rp_amp, kind='cubic', fill_value='extrapolate')(t_uni)
        qrs_uni = np.full_like(rr_uni, qrs_amp_std)
        hr_uni  = np.full_like(rr_uni, mean_hr)
        tri_uni = np.full_like(rr_uni, tri_index)
        pnn_uni = np.full_like(rr_uni, pnn50)

        n = len(rr_uni)
        def pad_or_crop(arr):
            if n >= SEG_LEN:
                s = (n - SEG_LEN) // 2
                return arr[s:s+SEG_LEN]
            else:
                pad = SEG_LEN - n
                return np.pad(arr, (pad//2, pad-pad//2), 'edge')

        return np.stack([
            pad_or_crop(rr_uni),
            pad_or_crop(rp_uni),
            pad_or_crop(qrs_uni),
            pad_or_crop(hr_uni),
            pad_or_crop(tri_uni),
            pad_or_crop(pnn_uni)
        ], axis=0).astype(np.float32)

    except:
        return None

# ============================================================
# LOAD MODEL
# ============================================================
def load_model(model_path=MODEL_PATH):
    model = MultiScaleSEResNet(in_channels=6)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print(f"✓ Model loaded from {model_path}")
    return model

# ============================================================
# INFERENCE
# ============================================================
def predict(ecg_segment, model, threshold=THRESHOLD):
    """
    Input  : ecg_segment — numpy array (12000,)
             raw ECG 60s @ 200Hz (no preprocessing needed)
    Output : label ('Normal' or 'AH'), probability
    """
    # Step 1: Bandpass 8-50Hz
    ecg_filtered = bandpass_filter(ecg_segment)

    # Step 2: Z-score normalization
    ecg_normalized = zscore_normalize(ecg_filtered)

    # Step 3: Extract 6 features
    features = extract_6_features(ecg_normalized)
    if features is None:
        return None, None

    # Step 4: Predict
    x = torch.Tensor(features).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = model(x).item()

    pred  = 1 if prob >= threshold else 0
    label = "AH" if pred == 1 else "Normal"
    return label, prob

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    model = load_model()

    # Example usage:
    # ecg_segment = np.load('your_ecg_segment.npy')  # shape: (12000,)
    # label, prob = predict(ecg_segment, model)
    # print(f"Prediction: {label} | Probability: {prob:.4f}")

    print("\nInference ready.")
    print("Input  : raw ECG segment (12000 samples, 60s @ 200Hz)")
    print("Output : 'Normal' or 'AH' + probability")